# 08 — Weighted XGBoost E + LR E/Position-bin 50:50 검증 및 제출 후보

- **핫스팟 사용 안 함**. 시윤님 `03_hotspot_bins.ipynb`의 `pipeline_E_position_LR` 정의를 재현.
- 기존 06 노트북의 **팀 고정 홀드아웃** 4,960/1,241 재사용.
- 개발셋 Group CV에서 기존 08 가중치 XGB + LR E 대비 신규 XGB + LR E/Position 비교.
- 동일 홀드아웃에서 비교. **CV 개선 > 0 및 홀드아웃 하락 -0.02 이내**면 통과 후보.
- 통과 후 `RUN_FULL_CV=True`, `SAVE_ARTIFACTS=True`, `SAVE_SUBMISSION=True`로 전체 Train OOF와 XGB 단독 Test 확률을 생성하고, **동일 설정의 LR 위치 구간 OOF/Test**과 50:50 제출 파일 생성.
- 홀드아웃 점수 확인 후 파라미터나 피처를 재선택하지 마세요. 이미 재사용된 홀드아웃이므로 독립적인 최종 평가가 아닙니다.

**실행:** 프로젝트의 VS Code 노트북 커널에서 위에서 아래로 실행. 첫 실행은 `RUN_FINAL=False`.

In [10]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
RUN_FULL_CV = True    # 팀장 승인 후 True: 전체 OOF/Test 생성 (기존 정상 파일 재사용)
SAVE_ARTIFACTS = True # True: XGB 단독/LR/앙상블 예측 파일 저장
SAVE_SUBMISSION = True # True: 최종 제출 CSV 저장
print('ROOT', ROOT, '| XGBoost', __import__('xgboost').__version__)
print('홀드아웃 스크립트:', manifest_path)

ROOT C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | XGBoost 3.4.1
홀드아웃 스크립트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\scripts\fixed_holdout_manifest.py


In [2]:
def select_xgb_device():
    try:
        probe = XGBClassifier(n_estimators=1, max_depth=1, tree_method='hist', device='cuda',
                              objective='binary:logistic', eval_metric='logloss', verbosity=0)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            probe.fit(np.array([[0.],[1.],[2.],[3.]],dtype=np.float32), np.array([0,0,1,1]))
        actual = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        if str(actual).startswith('cuda'):
            return 'cuda'
    except Exception as exc:
        print('CUDA 확인 실패:', repr(exc))
    return 'cpu'

DEVICE = select_xgb_device()
print('XGBoost device:', DEVICE)

XGBoost device: cuda


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert len(le.classes_) == 26
# 공통 홀드아웃 먼저 읽기: CV는 개발 데이터에서만 실행합니다.
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx) + len(hold_idx) == len(y)
assert not np.intersect1d(dev_idx, hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)  # 팀 함수 변경 금지
print('train', X_raw.shape, 'test', X_test_raw.shape, 'classes', len(le.classes_))

def make_E(X):
    # 시윤님 02_features E와 동일한 피처 순서/변환
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()
    return out

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns)
assert X_E.shape[1] == 9174, f'E 피처 개수 불일치: {X_E.shape[1]}'
print('E', X_E.shape, X_test_E.shape)
X_dev_E = X_E.iloc[dev_idx].reset_index(drop=True)


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
train (6201, 4384) test (2546, 4384) classes 26
E (6201, 9174) (2546, 9174)


In [4]:
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])



In [5]:
# 동일 개발셋 Fold에서 기존 08 vs 신규 50:50 앙상블 CV 비교
K = len(le.classes_)
common = dict(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.8,
              colsample_bytree=0.8, objective='multi:softprob', num_class=K,
              eval_metric='mlogloss', tree_method='hist', device=DEVICE,
              random_state=SEED, n_jobs=-1)
xgb = XGBClassifier(**common)
lr_E = LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000, solver='lbfgs')

def weight_array(y_part):
    counts = np.bincount(np.asarray(y_part, dtype=int), minlength=K)
    assert (counts > 0).all()
    w = 1.0 / np.sqrt(counts[np.asarray(y_part, dtype=int)])
    return (w / w.mean()).astype(np.float32)

def weighted_fit_params(y_part):
    return {'sample_weight': weight_array(y_part)}

def checked(p,n):
    p=np.asarray(p,dtype=np.float64)
    assert p.shape == (n,K) and np.isfinite(p).all()
    assert np.allclose(p.sum(axis=1),1,atol=1e-4)
    return p

X_lr_dev = pd.DataFrame.sparse.from_spmatrix(csr_matrix(X_dev_E.to_numpy()),index=X_dev_E.index,columns=X_dev_E.columns)
cv_xgb = run_cv(xgb, X_dev_E, y_dev, folds=folds, fit_params_fn=weighted_fit_params)
cv_lr_E = run_cv(lr_E, X_lr_dev, y_dev, folds=folds)
cv_lr_pos = run_cv(pipeline_E_position_LR, X_dev_raw, y_dev, folds=folds)
px = checked(cv_xgb['oof_proba'],len(y_dev))
pe = checked(cv_lr_E['oof_proba'],len(y_dev))
pp = checked(cv_lr_pos['oof_proba'],len(y_dev))
old_oof = checked((px+pe)/2,len(y_dev))
new_oof = checked((px+pp)/2,len(y_dev))

def fold_scores(proba, labels, folds_):
    return np.array([f1_score(labels[va],proba[va].argmax(axis=1),average='macro',zero_division=0) for _,va in folds_])
old_scores=fold_scores(old_oof,y_dev,folds)
new_scores=fold_scores(new_oof,y_dev,folds)
cv_delta=float(new_scores.mean()-old_scores.mean())
display(pd.DataFrame([
    {'model':'기존 08: weighted XGB E + LR E','cv_mean':old_scores.mean(),'cv_std':old_scores.std()},
    {'model':'신규: weighted XGB E + LR E+position','cv_mean':new_scores.mean(),'cv_std':new_scores.std()},
]))
print('개발 CV 변화:',round(cv_delta,6))


c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [19:29:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold 1: Macro F1 = 0.5049
fold 2: Macro F1 = 0.4753
fold 3: Macro F1 = 0.4806
fold 4: Macro F1 = 0.4810
fold 5: Macro F1 = 0.5048
CV Macro F1 = 0.4893 (std 0.0128)
fold 1: Macro F1 = 0.5025
fold 2: Macro F1 = 0.4644
fold 3: Macro F1 = 0.4740
fold 4: Macro F1 = 0.4757
fold 5: Macro F1 = 0.4889
CV Macro F1 = 0.4811 (std 0.0132)
fold 1: Macro F1 = 0.5133
fold 2: Macro F1 = 0.4652
fold 3: Macro F1 = 0.4629
fold 4: Macro F1 = 0.4909
fold 5: Macro F1 = 0.4982
CV Macro F1 = 0.4861 (std 0.0194)


,model,cv_mean,cv_std
0,기존 08: weighted XGB E + LR E,0.501080,0.014111
1,신규: weighted XGB E + LR E+position,0.506699,0.017581


개발 CV 변화: 0.005619


In [6]:
# 동일한 팀 고정 홀드아웃 단 1회 비교
assert len(dev_idx)==4960 and len(hold_idx)==1241
raw_groups=pd.util.hash_pandas_object(X_raw.fillna('<MISSING>').astype(str),index=False).to_numpy()
assert not set(raw_groups[dev_idx]) & set(raw_groups[hold_idx])
X_dev_E_hold, X_hold_E = X_E.iloc[dev_idx],X_E.iloc[hold_idx]
X_dev_raw_hold,X_hold_raw=X_raw.iloc[dev_idx],X_raw.iloc[hold_idx]
y_hold=np.asarray(y)[hold_idx]
mx=clone(xgb).fit(X_dev_E_hold,y_dev,sample_weight=weight_array(y_dev))
me=clone(lr_E).fit(csr_matrix(X_dev_E_hold.to_numpy()),y_dev)
mp=clone(pipeline_E_position_LR).fit(X_dev_raw_hold,y_dev)
for m in (mx,me,mp): assert np.array_equal(m.classes_,np.arange(K))
p_x=checked(mx.predict_proba(X_hold_E),len(y_hold))
p_e=checked(me.predict_proba(csr_matrix(X_hold_E.to_numpy())),len(y_hold))
p_p=checked(mp.predict_proba(X_hold_raw),len(y_hold))
old_hold=float(f1_score(y_hold,((p_x+p_e)/2).argmax(axis=1),average='macro',zero_division=0))
new_hold=float(f1_score(y_hold,((p_x+p_p)/2).argmax(axis=1),average='macro',zero_division=0))
hold_delta=new_hold-old_hold
candidate_pass=(cv_delta>0) and (hold_delta>=-0.02-1e-12)
display(pd.DataFrame([
    {'candidate':'기존 08: XGB E + LR E','dev_cv_mean':old_scores.mean(),'holdout_macro_f1':old_hold},
    {'candidate':'신규: XGB E + LR E+position','dev_cv_mean':new_scores.mean(),'holdout_macro_f1':new_hold},
]))
print(f'CV 변화 {cv_delta:+.5f} | 홀드아웃 변화 {hold_delta:+.5f}')
print('규칙 판단:', '통과 후보 (팀장 승인 필요)' if candidate_pass else '보류')


,candidate,dev_cv_mean,holdout_macro_f1
0,기존 08: XGB E + LR E,0.501080,0.499884
1,신규: XGB E + LR E+position,0.506699,0.504429


CV 변화 +0.00562 | 홀드아웃 변화 +0.00455
규칙 판단: 통과 후보 (팀장 승인 필요)


## 최종 산출물 생성 (팀장 승인 후)

아래 셀은 `RUN_FINAL=True`이고 검증 통과일 때만 실행됩니다. 전체 6,201행의 팀 공통 Group 5-Fold OOF와 XGBoost 전체 Train 재학습 Test 예측을 저장합니다. **LR E+Position Test 예측은 시윤님이 별도로 올린 파일을 사용**합니다. 파일이 없으면 제출 생성은 중단합니다. 전체 Train OOF와 고정 홀드아웃 점수는 서로 다른 평가입니다.

## 7. 전체 Train OOF / Test 예측 (07번과 같은 True/False 방식)

팀장 승인 후 **위 설정 셀에서 `RUN_FULL_CV=True`, `SAVE_ARTIFACTS=True`, `SAVE_SUBMISSION=True`**로 변경하세요. 검증된 모델 설정을 변경하지 않습니다. 이미 정상 저장된 예측은 검증 후 재사용합니다.

In [11]:
# 최종 예측 생성: 기존 결과가 있으면 재사용, 없으면 필요한 모델만 학습
import os, tempfile
PRED = ROOT / 'experiments' / 'preds'
PRED.mkdir(parents=True, exist_ok=True)
XGB_STEM = 'A_model__xgb_E_depth3_invsqrt_position_ensemble_1110'
LR_STEM = 'H_data__E_position_bin'
ENS_STEM = 'A_model__ensemble_E_xgb_invsqrt_position_lr_5050_1110'
N, NT = len(y), len(test)

def valid_proba(a, rows, title):
    a = np.asarray(a)
    if a.shape != (rows, K) or not np.isfinite(a).all() or (a < -1e-6).any() or not np.allclose(a.sum(axis=1), 1, atol=1e-4):
        raise ValueError(f'{title}: 확률 배열 형상/합계 오류 {a.shape}')
    return a

def load_saved(stem, kind, rows):
    path = PRED / f'{stem}__{kind}.npy'
    if not path.exists(): return None
    a = valid_proba(np.load(path, allow_pickle=False), rows, path.name)
    meta = PRED / f'{stem}__meta.json'
    if meta.exists():
        old = json.loads(meta.read_text(encoding='utf-8'))
        if 'classes' in old and list(map(str, old['classes'])) != list(map(str, le.classes_)):
            raise ValueError(f'{meta.name}: 클래스 순서 불일치')
    print('기존 정상 파일 재사용:', path.name)
    return a

if not RUN_FULL_CV:
    print('전체 OOF/Test 생성 대기: RUN_FULL_CV=True 설정')
else:
    if not candidate_pass:
        raise RuntimeError('검증 통과 여부를 확인하세요: candidate_pass=False')
    full_folds = list(get_folds(y, X=X_raw, classes=le.classes_))
    coverage = np.zeros(N, dtype=int)
    for tr, va in full_folds: coverage[va] += 1
    assert len(full_folds)==5 and np.all(coverage==1), '전체 OOF Fold coverage 확인 필요'
    xgb_oof = load_saved(XGB_STEM, 'oof', N)
    xgb_test = load_saved(XGB_STEM, 'test', NT)
    if xgb_oof is None:
        print('XGBoost 단독 전체 OOF 생성 중')
        cv_full_xgb = run_cv(xgb, X_E, y, folds=full_folds, fit_params_fn=weighted_fit_params)
        xgb_oof = valid_proba(cv_full_xgb['oof_proba'], N, 'XGB OOF')
    if xgb_test is None:
        print('XGBoost 단독 전체 Train 재학습 → Test 생성 중')
        final_xgb = clone(xgb).fit(X_E, y, sample_weight=weight_array(y))
        assert np.array_equal(final_xgb.classes_, np.arange(K))
        xgb_test = valid_proba(final_xgb.predict_proba(X_test_E), NT, 'XGB Test')
    lr_oof = load_saved(LR_STEM, 'oof', N)
    lr_test = load_saved(LR_STEM, 'test', NT)
    if lr_oof is None or lr_test is None:
        print('LR E+위치 구간 5-Fold OOF/Test 생성 중 (핫스팟 미사용)')
        lr_oof_new = np.zeros((N,K), dtype=np.float64) if lr_oof is None else None
        lr_test_parts = [] if lr_test is None else None
        for i,(tr,va) in enumerate(full_folds,1):
            m = clone(pipeline_E_position_LR).fit(X_raw.iloc[tr], np.asarray(y)[tr])
            assert np.array_equal(m.classes_, np.arange(K)), f'LR Fold {i}: 클래스 순서 오류'
            if lr_oof_new is not None:
                lr_oof_new[va] = valid_proba(m.predict_proba(X_raw.iloc[va]),len(va),f'LR OOF {i}')
            if lr_test_parts is not None:
                lr_test_parts.append(valid_proba(m.predict_proba(X_test_raw),NT,f'LR Test {i}'))
            print(f'LR Fold {i}/5 완료')
        if lr_oof is None: lr_oof = valid_proba(lr_oof_new,N,'LR OOF')
        if lr_test is None: lr_test = valid_proba(np.mean(lr_test_parts,axis=0),NT,'LR Test')
    ens_oof = valid_proba((xgb_oof + lr_oof)/2,N,'Ensemble OOF')
    ens_test = valid_proba((xgb_test + lr_test)/2,NT,'Ensemble Test')
    print('XGB OOF Macro F1:', round(f1_score(y,xgb_oof.argmax(axis=1),average='macro',zero_division=0),6))
    print('LR OOF Macro F1:', round(f1_score(y,lr_oof.argmax(axis=1),average='macro',zero_division=0),6))
    print('앙상블 OOF Macro F1:', round(f1_score(y,ens_oof.argmax(axis=1),average='macro',zero_division=0),6))


기존 정상 파일 재사용: A_model__xgb_E_depth3_invsqrt_position_ensemble_1110__oof.npy
기존 정상 파일 재사용: A_model__xgb_E_depth3_invsqrt_position_ensemble_1110__test.npy
기존 정상 파일 재사용: H_data__E_position_bin__oof.npy
기존 정상 파일 재사용: H_data__E_position_bin__test.npy
XGB OOF Macro F1: 0.495409
LR OOF Macro F1: 0.50426
앙상블 OOF Macro F1: 0.516561


## 8. 팀 표준 예측 파일 저장 — `SAVE_ARTIFACTS=True`

이미 저장된 파일이 정상이고 현재 계산 결과와 같으면 재사용합니다. **기존 파일을 덮어쓰지 않습니다.** 중간에 실행이 끊겨도 다시 실행할 수 있습니다.

In [12]:
def save_array(stem, kind, arr):
    path = PRED / f'{stem}__{kind}.npy'
    arr = valid_proba(arr, len(arr), path.name)
    if path.exists():
        existing = valid_proba(np.load(path,allow_pickle=False),len(arr),path.name)
        if not np.allclose(existing,arr,atol=1e-5):
            raise ValueError(f'기존 파일과 현재 계산 결과가 다릅니다 (자동 덮어쓰기 방지): {path.name}')
        print('저장 완료 (기존 동일 파일):',path.name)
        return
    fd, tmp = tempfile.mkstemp(dir=PRED,suffix='.npy'); os.close(fd)
    try:
        np.save(tmp,np.asarray(arr,dtype=np.float32))
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp): os.unlink(tmp)
    print('저장 완료:',path.name)

def save_meta(stem, extra):
    path = PRED / f'{stem}__meta.json'
    obj = {'classes':list(map(str,le.classes_)),'train_rows':N,'test_rows':NT,'folds':5,**extra}
    if path.exists():
        existing=json.loads(path.read_text(encoding='utf-8'))
        if 'classes' in existing and list(map(str,existing['classes']))!=obj['classes']:
            raise ValueError(f'기존 메타 클래스 불일치: {path.name}')
        print('기존 메타 유지:',path.name)
    else:
        path.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
        print('메타 저장:',path.name)

if not SAVE_ARTIFACTS:
    print('저장 대기: SAVE_ARTIFACTS=True 설정')
elif not RUN_FULL_CV:
    print('저장 대기: RUN_FULL_CV=True 설정 필요')
else:
    for stem, oof, tst, extra in [
        (XGB_STEM,xgb_oof,xgb_test,{'model':'weighted XGB E depth3 250 trees','sample_weight':'inverse sqrt frequency'}),
        (LR_STEM,lr_oof,lr_test,{'model':'LR E+position-bin','hotspots_used':False,'bin_size':50,'min_patients':5,'C':0.1}),
        (ENS_STEM,ens_oof,ens_test,{'model':'XGB E + LR E/position-bin 50:50','hotspots_used':False,'xgb_source':XGB_STEM,'lr_source':LR_STEM}),
    ]:
        save_array(stem,'oof',oof)
        save_array(stem,'test',tst)
        save_meta(stem,extra)


저장 완료 (기존 동일 파일): A_model__xgb_E_depth3_invsqrt_position_ensemble_1110__oof.npy
저장 완료 (기존 동일 파일): A_model__xgb_E_depth3_invsqrt_position_ensemble_1110__test.npy
기존 메타 유지: A_model__xgb_E_depth3_invsqrt_position_ensemble_1110__meta.json
저장 완료 (기존 동일 파일): H_data__E_position_bin__oof.npy
저장 완료 (기존 동일 파일): H_data__E_position_bin__test.npy
기존 메타 유지: H_data__E_position_bin__meta.json
저장 완료 (기존 동일 파일): A_model__ensemble_E_xgb_invsqrt_position_lr_5050_1110__oof.npy
저장 완료 (기존 동일 파일): A_model__ensemble_E_xgb_invsqrt_position_lr_5050_1110__test.npy
기존 메타 유지: A_model__ensemble_E_xgb_invsqrt_position_lr_5050_1110__meta.json


## 9. 앙상블 제출 CSV — `SAVE_SUBMISSION=True`

`sample_submission.csv`의 ID와 행 순서를 확인한 뒤 저장합니다. 이미 동일한 제출 파일이 있으면 재사용합니다.

In [13]:
if not SAVE_SUBMISSION:
    print('제출 CSV 저장 대기: SAVE_SUBMISSION=True 설정')
elif not (RUN_FULL_CV and SAVE_ARTIFACTS):
    print('제출 대기: RUN_FULL_CV=True, SAVE_ARTIFACTS=True 설정 필요')
else:
    from src.config import ID_COL, TARGET
    sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    if len(sample)!=NT or not np.array_equal(sample[ID_COL].to_numpy(),test[ID_COL].to_numpy()):
        raise ValueError('sample_submission과 Test ID/행 순서 불일치')
    labels = le.inverse_transform(ens_test.argmax(axis=1))
    submission = sample.copy()
    submission[TARGET] = labels
    outdir = ROOT / 'submissions'; outdir.mkdir(parents=True,exist_ok=True)
    out = outdir / f'{ENS_STEM}__submission.csv'
    if out.exists():
        prior = pd.read_csv(out)
        if not prior.equals(submission):
            raise ValueError('기존 제출 CSV가 현재 예측과 다릅니다. 확인 없이 덮어쓰지 않습니다.')
        print('기존 제출 CSV 재사용:',out)
    else:
        submission.to_csv(out,index=False,encoding='utf-8-sig')
        print('제출 CSV 저장:',out)
    print('=== 최종 완료 === | 행:',len(submission),'| 클래스:',K)
    display(submission.head())


기존 제출 CSV 재사용: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\submissions\A_model__ensemble_E_xgb_invsqrt_position_lr_5050_1110__submission.csv
=== 최종 완료 === | 행: 2546 | 클래스: 26


,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,STES
2,TEST_0002,THCA
3,TEST_0003,LGG
4,TEST_0004,LUSC


## 10. PR 안내

PR base: `dev`. 가중치 XGBoost E 단독 OOF/Test, **핫스팟 없는 E + 위치 구간 LR** OOF/Test, 50:50 앙상블 OOF/Test 및 제출 CSV를 포함합니다. 개발 CV **0.50108 → 0.50670**, 홀드아웃 **0.49988 → 0.50443** (기존 검증 결과). 전체 OOF 점수는 홀드아웃 점수와 구분해서 기록하세요.